# HotpotQA v2: global retrieval + HippoRAG2 ablations
Restore an existing construction ZIP. Compare dense retrieval, Entity-KG, Entity-Event-KG and Full-KG with the same Qwen3.5-2B reader. Uses upstream HippoRAG2 PageRank, MiniLM CPU embeddings, FAISS and LLM fact filtering. This is a small-model adaptation, **not** exact paper reproduction. No graph reconstruction here.

Select a GPU runtime first. Checkpoints/cache and the input ZIP will be saved in your Google Drive. Re-run identical cells to resume; do not delete the run folder. Three questions remain only a smoke test. See `HOTPOTQA_V2.md` for larger seeded samples.

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
import requests
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode != 0:
    raise RuntimeError('Select Runtime > Change runtime type > GPU before continuing')
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/AutoSchemaKG/hotpotqa_v2_smoke')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_ZIP = RUN_ROOT / 'construction.zip'
OUTPUT_DIR = RUN_ROOT / 'benchmark'
MODEL_ID = 'Qwen/Qwen3.5-2B'
EMBEDDING_MODEL = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
PORT = 8000
CONTEXT_LENGTH = 4096
print('Persistent experiment directory:', RUN_ROOT)

## 1. Restore your input
Upload the construction/evaluated ZIP, not `content.zip`. The input must contain GraphML, corpus, QA manifest and extraction JSON. Existing `construction.zip` is reused without overwriting.

In [ ]:
if not INPUT_ZIP.exists():
    uploaded = files.upload()
    candidates = [Path(name).resolve() for name in uploaded if name.lower().endswith('.zip')]
    if len(candidates) != 1:
        raise ValueError('Upload exactly one construction ZIP')
    shutil.copy2(candidates[0], INPUT_ZIP)
print('Using:', INPUT_ZIP, 'bytes:', INPUT_ZIP.stat().st_size)

## 2. Clone once and pin code/model revisions
On resume this keeps the same git commit. To evaluate changed code or models, use a new `RUN_ROOT`. The upgraded files must be pushed to your GitHub before using this clone cell.

In [ ]:
REPO_DIR = Path('/content/SmallScaledAutoSchemaKG_v2')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git', str(REPO_DIR)], check=True)
revision_file = RUN_ROOT / 'revisions.json'
if revision_file.exists():
    revisions = json.loads(revision_file.read_text())
    if revisions['model'] != MODEL_ID or revisions['embedding'] != EMBEDDING_MODEL:
        raise ValueError('Model changed: use a NEW RUN_ROOT')
    status = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True).strip()
    if status:
        raise RuntimeError('Clone has local edits; preserve them before switching revision')
    subprocess.run(['git', 'checkout', '--detach', revisions['git_commit']], cwd=REPO_DIR, check=True)
else:
    def model_revision(model):
        response = requests.get(f'https://huggingface.co/api/models/{model}', timeout=30)
        response.raise_for_status()
        return response.json()['sha']
    revisions = {'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
                 'model': MODEL_ID, 'model_revision': model_revision(MODEL_ID),
                 'embedding': EMBEDDING_MODEL, 'embedding_revision': model_revision(EMBEDDING_MODEL)}
    if not (REPO_DIR / 'scripts/run_hotpotqa_benchmark.py').exists():
        raise RuntimeError('v2 files are not present on this GitHub revision yet')
    revision_file.write_text(json.dumps(revisions, indent=2))
os.chdir(REPO_DIR)
print(json.dumps(revisions, indent=2))

## 3. Install isolated environments
QA embeddings run on CPU; only vLLM uses GPU. Separate environments avoid the Torch/TorchAudio CUDA mismatch encountered in the earlier notebook. No changes to Colab's preinstalled TorchAudio/TorchVision are required. First installation can take several minutes. Locks are saved to Drive for the next runtime.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
QA_ENV = '/content/autoschema_qa_env'
VLLM_ENV = '/content/autoschema_vllm_env'
QA_PY = QA_ENV + '/bin/python'
VLLM_PY = VLLM_ENV + '/bin/python'
for env in (QA_ENV, VLLM_ENV):
    if not Path(env, 'bin/python').exists():
        subprocess.run(['uv', 'venv', '--python', '3.12', env], check=True)
qa_lock = RUN_ROOT / 'qa_requirements.lock.txt'
subprocess.run(['uv', 'pip', 'install', '--python', QA_PY, '--torch-backend=cpu', '-r',
                str(qa_lock) if qa_lock.exists() else 'requirements-hotpotqa-v2.txt'], check=True)
if not qa_lock.exists():
    qa_lock.write_text(subprocess.check_output(['uv', 'pip', 'freeze', '--python', QA_PY], text=True))
vllm_lock = RUN_ROOT / 'vllm_requirements.lock.txt'
vllm_args = ['-r', str(vllm_lock)] if vllm_lock.exists() else ['--pre', 'vllm']
subprocess.run(['uv', 'pip', 'install', '--python', VLLM_PY, '--torch-backend=auto'] + vllm_args, check=True)
if not vllm_lock.exists():
    vllm_lock.write_text(subprocess.check_output(['uv', 'pip', 'freeze', '--python', VLLM_PY], text=True))
subprocess.run([QA_PY, '-u', 'scripts/run_hotpotqa_benchmark.py', str(INPUT_ZIP), '--inspect-only'], check=True)

## 4. Start local Qwen
Wait for `/v1/models` to be ready. If the server exits, the cell prints its log tail. Keep this runtime alive during the benchmark. After a runtime reset, restart this server; checkpoints remain in Drive.

In [ ]:
LOG_PATH = RUN_ROOT / 'qwen_vllm.log'
def ready():
    try:
        response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
        if response.ok:
            models = response.json()['data']
            if not any(m['id'] == MODEL_ID for m in models):
                raise RuntimeError('Port is occupied by another model; change PORT')
            return True
    except requests.RequestException:
        return False
    return False
if not ready():
    server_log = open(LOG_PATH, 'a', encoding='utf-8')
    server = subprocess.Popen([VLLM_ENV + '/bin/vllm', 'serve', MODEL_ID,
        '--revision', revisions['model_revision'], '--host', '127.0.0.1', '--port', str(PORT),
        '--dtype', 'half', '--max-model-len', str(CONTEXT_LENGTH), '--max-num-seqs', '1',
        '--gpu-memory-utilization', '0.80', '--language-model-only'],
        stdout=server_log, stderr=subprocess.STDOUT)
    started = time.monotonic()
    while time.monotonic() - started < 1200:
        if server.poll() is not None:
            server_log.flush()
            print(LOG_PATH.read_text(errors='replace')[-12000:])
            raise RuntimeError('vLLM stopped; inspect log above')
        if ready():
            break
        print(f'Waiting for Qwen: {time.monotonic() - started:.0f}s', flush=True)
        time.sleep(10)
    else:
        raise TimeoutError(f'Server not ready. Inspect {LOG_PATH}; do not start another copy')
print('Local Qwen is ready')

## 5. Run / resume all four methods
The script searches **all passages in the supplied corpus**, not only the 10 contexts of a question. It saves each completed question before continuing and reports ETA per method. EM/F1 and document support recall are on a 0–1 scale. To change parameters, use a new `OUTPUT_DIR`; old checkpoints are never silently mixed.

In [ ]:
command = [QA_PY, '-u', 'scripts/run_hotpotqa_benchmark.py', str(INPUT_ZIP),
    '--output-dir', str(OUTPUT_DIR), '--variants', 'dense', 'entity', 'entity_event', 'full',
    '--model', MODEL_ID, '--model-revision', revisions['model_revision'],
    '--base-url', f'http://127.0.0.1:{PORT}/v1',
    '--embedding-model', EMBEDDING_MODEL, '--embedding-revision', revisions['embedding_revision'],
    '--embedding-device', 'cpu', '--top-edges', '30', '--top-passages', '5',
    '--ppr-alpha', '0.9', '--passage-weight', '0.9', '--context-length', str(CONTEXT_LENGTH)]
subprocess.run(command, check=True)

## 6. Read results and save a portable ZIP
`summary.json` marks whether each method is complete. Partial averages are not final results. The archive includes the original construction ZIP, per-question evidence/predictions, configuration, environment locks, model revisions and embedding cache. Keep both this archive and the executed notebook. No model weights are included.

In [ ]:
summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    for method, data in summary['methods'].items():
        print(method, 'completed:', data['completed'], 'complete:', data['complete'])
archive = shutil.make_archive('/content/autoschemakg_hotpotqa_v2', 'zip', RUN_ROOT)
print('Checkpoint directory remains on Drive:', RUN_ROOT)
files.download(archive)